<a href="https://colab.research.google.com/github/DannyCerort/pln-proyecto/blob/main/Proyecto_PLN_Colab_LIMPIO_METRICAS_CORREGIDAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto PLN: Extracción de fechas desde OBSERVACION

Este notebook implementa un pipeline de Procesamiento de Lenguaje Natural para extraer información temporal desde la columna `OBSERVACION` de una base de facturación logística.

El objetivo es identificar mes y año de prestación del servicio, construir una fecha estructurada y calcular el desfase frente a la fecha de factura.

## 1. Cargar archivo Excel

Ejecuta la siguiente celda y sube el archivo:

`Unificación de Bases de Datos.xlsx`

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Unificación de Bases de Datos.xlsx to Unificación de Bases de Datos.xlsx


## 2. Importar librerías

In [2]:
import pandas as pd
import re
import unicodedata
from datetime import datetime
from pathlib import Path
import numpy as np

In [12]:
import re
import pandas as pd
import numpy as np

# =====================================================
# Diccionario de meses
# =====================================================

meses = {

    'enero': 1,
    'ene': 1,

    'febrero': 2,
    'feb': 2,

    'marzo': 3,
    'mar': 3,

    'abril': 4,
    'abr': 4,

    'mayo': 5,
    'may': 5,

    'junio': 6,
    'jun': 6,

    'julio': 7,
    'jul': 7,

    'agosto': 8,
    'ago': 8,

    'septiembre': 9,
    'setiembre': 9,
    'sep': 9,
    'sept': 9,

    'octubre': 10,
    'oct': 10,

    'noviembre': 11,
    'nov': 11,

    'diciembre': 12,
    'dic': 12

}

# =====================================================
# Patrones regex
# =====================================================

patron_meses = (
    r'\\b(' +
    '|'.join(meses.keys()) +
    r')\\b'
)

patron_anio = r'\\b(20\\d{2})\\b'

print("Patrones cargados correctamente")
print(patron_meses)

Patrones cargados correctamente
\\b(enero|ene|febrero|feb|marzo|mar|abril|abr|mayo|may|junio|jun|julio|jul|agosto|ago|septiembre|setiembre|sep|sept|octubre|oct|noviembre|nov|diciembre|dic)\\b


## 3. Leer la base de datos

La base contiene la hoja `Unificación BD`, la columna textual `OBSERVACION` y la fecha de factura en la columna `FECHA`.

In [3]:
archivo = 'Unificación de Bases de Datos.xlsx'
hoja = 'Unificación BD'

df = pd.read_excel(archivo, sheet_name=hoja)

print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])
df.head()

Filas: 8011
Columnas: 27


,EMPRESA,AÑO DOCUMENTO,MES,subtipo,Numero documento,FECHA,ID_NIT,ID_NOMBRE_CLIENTE,SUB_PED,PEDIDO,...,%RETENCION,COD RETENCION,RETEFUENTE,VAL TOTAL,ORDEN DE COMPRA,OBSERVACION,est Dian,Usuario,vendedor,VENDEDOR2
0,INTERSERVICE,2024,NaN,10,FE13501,2024-01-02,1,CLIENTE_1,0,0,...,1,16,33698.26,3322176.66,4824117 ...,SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVI...,1,TREYES,1,0
1,INTERSERVICE,2024,1.0,10,FE13502,2024-01-02,1,CLIENTE_1,0,0,...,1,16,33698.26,3322176.66,4824136 ...,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,1,TREYES,1,0
2,INTERSERVICE,2024,1.0,10,FE13503,2024-01-02,2,CLIENTE_2,5,2052,...,1,16,34431.78,3394491.46,0 ...,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,1,sncastaneda,1,0
3,INTERSERVICE,2024,1.0,10,FE13504,2024-01-02,3,CLIENTE_3,0,0,...,1,16,21460.00,2115655.56,9400069513 ...,SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE ...,1,sncastaneda,1,0
4,INTERSERVICE,2024,1.0,10,FE13505,2024-01-02,4,CLIENTE 4,5,2053,...,2,16,99484.14,9807743.43,0 ...,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,1,sncastaneda,1,0


## 4. Normalización de texto

Se convierte el texto a minúsculas, se eliminan tildes, caracteres especiales y espacios duplicados.

In [4]:
def normalizar_texto(texto):
    if pd.isna(texto):
        return ''
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'[^a-z0-9\s/\-]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['observacion_normalizada'] = df['OBSERVACION'].apply(normalizar_texto)
df[['OBSERVACION', 'observacion_normalizada']].head(10)

,OBSERVACION,observacion_normalizada
0,SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVI...,servicio de transporte terrestre de carga livi...
1,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...
2,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,servicio de mensajeria fija del mes de diciemb...
3,SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE ...,servicio de mensajeria y transporte terrestre ...
4,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,servicio de mensajeria fija del mes de diciemb...
5,SERVICIO DE MENSAJERIA PRESTADO DURANTE EL MES...,servicio de mensajeria prestado durante el mes...
6,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...
7,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...
8,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...
9,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...


## 4.1 Tokenización del texto

Después de normalizar la columna `OBSERVACION`, se aplica tokenización para separar cada observación en palabras o unidades de análisis. Esto permite identificar con mayor claridad los meses, años y patrones temporales dentro del texto libre.

La tokenización es una etapa central del Procesamiento de Lenguaje Natural porque transforma una cadena de texto completa en una lista de términos analizables.

In [5]:
# Tokenización básica usando expresiones regulares
def tokenizar_texto(texto):
    if pd.isna(texto) or texto == '':
        return []
    # Extrae palabras y números como tokens
    tokens = re.findall(r'\b[a-zA-Z0-9]+\b', str(texto))
    return tokens

# Aplicar tokenización sobre la observación normalizada
df['tokens_observacion'] = df['observacion_normalizada'].apply(tokenizar_texto)

# Cantidad de tokens por observación
df['cantidad_tokens'] = df['tokens_observacion'].apply(len)

# Visualizar ejemplo
df[['OBSERVACION', 'observacion_normalizada', 'tokens_observacion', 'cantidad_tokens']].head(10)

,OBSERVACION,observacion_normalizada,tokens_observacion,cantidad_tokens
0,SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVI...,servicio de transporte terrestre de carga livi...,"[servicio, de, transporte, terrestre, de, carg...",16
1,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",16
2,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,servicio de mensajeria fija del mes de diciemb...,"[servicio, de, mensajeria, fija, del, mes, de,...",9
3,SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE ...,servicio de mensajeria y transporte terrestre ...,"[servicio, de, mensajeria, y, transporte, terr...",11
4,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,servicio de mensajeria fija del mes de diciemb...,"[servicio, de, mensajeria, fija, del, mes, de,...",9
5,SERVICIO DE MENSAJERIA PRESTADO DURANTE EL MES...,servicio de mensajeria prestado durante el mes...,"[servicio, de, mensajeria, prestado, durante, ...",15
6,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",16
7,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",16
8,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",16
9,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",16


## 4.2 Análisis simple de tokens frecuentes

Este análisis permite revisar cuáles son las palabras más frecuentes dentro de las observaciones. Sirve para identificar términos relevantes como meses, servicios, periodos o expresiones comunes de facturación.

In [6]:
from collections import Counter

# Unificar todos los tokens en una sola lista
todos_los_tokens = [token for lista in df['tokens_observacion'] for token in lista]

# Contar frecuencia de tokens
frecuencia_tokens = Counter(todos_los_tokens)

# Crear DataFrame con los tokens más frecuentes
df_tokens_frecuentes = pd.DataFrame(
    frecuencia_tokens.most_common(30),
    columns=['token', 'frecuencia']
)

df_tokens_frecuentes

,token,frecuencia
0,de,14298
1,servicio,6857
2,carga,5627
3,liviana,5438
4,2025,4176
5,nacional,4166
6,mensajeria,3903
7,mes,3804
8,en,3438
9,oc,2975


## 4.3 Instalación y carga de SpaCy para español

En esta sección se instala y carga el modelo de español de SpaCy. Este modelo permite aplicar técnicas de PLN como tokenización lingüística, lematización y reconocimiento de entidades nombradas (NER).

In [7]:
# Instalación del modelo de español para SpaCy
!python -m spacy download es_core_news_sm

import spacy
nlp = spacy.load('es_core_news_sm')

print('Modelo de SpaCy cargado correctamente')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 25.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Modelo de SpaCy cargado correctamente


## 4.4 Lematización

La lematización permite convertir cada palabra a su forma base o lema. Esto ayuda a reducir variaciones del lenguaje y facilita el análisis posterior de las observaciones.

In [8]:
def lematizar_texto(texto):
    if pd.isna(texto) or texto == '':
        return ''
    doc = nlp(str(texto))
    lemas = [token.lemma_ for token in doc if not token.is_punct and not token.is_space]
    return ' '.join(lemas)

df['observacion_lematizada'] = df['observacion_normalizada'].apply(lematizar_texto)

df[['OBSERVACION', 'observacion_normalizada', 'observacion_lematizada']].head(10)

,OBSERVACION,observacion_normalizada,observacion_lematizada
0,SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVI...,servicio de transporte terrestre de carga livi...,servicio de transporte terrestre de carga livi...
1,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,servicio de mensajeria nacional de carga liviá...
2,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,servicio de mensajeria fija del mes de diciemb...,servicio de mensajeria fijo del mes de diciemb...
3,SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE ...,servicio de mensajeria y transporte terrestre ...,servicio de mensajeria y transporte terrestre ...
4,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,servicio de mensajeria fija del mes de diciemb...,servicio de mensajeria fijo del mes de diciemb...
5,SERVICIO DE MENSAJERIA PRESTADO DURANTE EL MES...,servicio de mensajeria prestado durante el mes...,servicio de mensajeria prestado durante el mes...
6,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,servicio de mensajeria nacional de carga liviá...
7,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,servicio de mensajeria nacional de carga liviá...
8,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,servicio de mensajeria nacional de carga liviá...
9,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,servicio de mensajeria nacional de carga livia...,servicio de mensajeria nacional de carga liviá...


## 4.5 Reconocimiento de Entidades Nombradas (NER)

El Reconocimiento de Entidades Nombradas permite identificar automáticamente entidades dentro del texto, como fechas, organizaciones, lugares o cantidades. Para este proyecto, el interés principal está en las entidades temporales asociadas a meses, años o periodos de prestación del servicio.

In [9]:
def extraer_entidades_ner(texto):
    if pd.isna(texto) or texto == '':
        return []
    doc = nlp(str(texto))
    entidades = []
    for ent in doc.ents:
        entidades.append((ent.text, ent.label_))
    return entidades

df['entidades_ner'] = df['observacion_normalizada'].apply(extraer_entidades_ner)

df[['OBSERVACION', 'entidades_ner']].head(20)

,OBSERVACION,entidades_ner
0,SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVI...,[]
1,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[]
2,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,[]
3,SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE ...,[]
4,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,[]
5,SERVICIO DE MENSAJERIA PRESTADO DURANTE EL MES...,"[(microsoft, ORG)]"
6,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[]
7,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[]
8,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[]
9,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[]


## 4.6 Extracción de entidades temporales con NER + reglas

Dado que los modelos NER generales no siempre reconocen correctamente meses escritos en observaciones operativas, se complementa el NER con reglas lingüísticas y expresiones regulares. Esta combinación permite mayor control y trazabilidad sobre la extracción temporal.

In [13]:
def entidades_temporales_reglas(texto):
    if pd.isna(texto) or str(texto).strip() == '':
        return []

    texto = str(texto).lower().strip()
    entidades = []

    # =====================================================
    # 1. Meses detectados por diccionario/reglas
    # =====================================================
    for match in re.finditer(patron_meses, texto):
        mes_texto = match.group(0)
        entidades.append({
            'texto': mes_texto,
            'tipo': 'MES',
            'valor': meses.get(mes_texto),
            'inicio': match.start(),
            'fin': match.end()
        })

    # =====================================================
    # 2. Años explícitos
    # =====================================================
    for match in re.finditer(patron_anio, texto):
        entidades.append({
            'texto': match.group(0),
            'tipo': 'ANIO',
            'valor': int(match.group(0)),
            'inicio': match.start(),
            'fin': match.end()
        })

    # =====================================================
    # 3. Patrones tipo mes/año: dic/2024, marzo-2025
    # =====================================================
    patron_mes_anio = r'\b(' + '|'.join(meses.keys()) + r')\s*[/\-]\s*(20\d{2})\b'

    for match in re.finditer(patron_mes_anio, texto):
        mes_txt = match.group(1)
        anio_txt = match.group(2)

        entidades.append({
            'texto': match.group(0),
            'tipo': 'MES_ANIO',
            'mes': meses.get(mes_txt),
            'anio': int(anio_txt),
            'inicio': match.start(),
            'fin': match.end()
        })

    # =====================================================
    # 4. Patrones numéricos tipo 12/2024 o 12-2024
    # =====================================================
    patron_mes_num_anio = r'\b(0?[1-9]|1[0-2])\s*[/\-]\s*(20\d{2})\b'

    for match in re.finditer(patron_mes_num_anio, texto):
        entidades.append({
            'texto': match.group(0),
            'tipo': 'MES_NUM_ANIO',
            'mes': int(match.group(1)),
            'anio': int(match.group(2)),
            'inicio': match.start(),
            'fin': match.end()
        })

    # =====================================================
    # 5. Rangos tipo enero a marzo / enero-marzo
    # =====================================================
    patron_rango_meses = (
        r'\b(' + '|'.join(meses.keys()) + r')'
        r'\s*(a|hasta|al|-)'
        r'\s*(' + '|'.join(meses.keys()) + r')\b'
    )

    for match in re.finditer(patron_rango_meses, texto):
        mes_inicio = match.group(1)
        mes_fin = match.group(3)

        entidades.append({
            'texto': match.group(0),
            'tipo': 'RANGO_MESES',
            'mes_inicio': meses.get(mes_inicio),
            'mes_fin': meses.get(mes_fin),
            'inicio': match.start(),
            'fin': match.end()
        })

    return entidades


df['entidades_temporales_reglas'] = df['observacion_normalizada'].apply(entidades_temporales_reglas)

# =====================================================
# Variables auxiliares para análisis
# =====================================================

def contar_entidades_tipo(entidades, tipo):
    return sum(1 for e in entidades if e.get('tipo') == tipo)

def tiene_entidad_temporal(entidades):
    return len(entidades) > 0

def es_ambiguo_temporal(entidades):
    meses_detectados = []

    for e in entidades:
        if e.get('tipo') == 'MES':
            meses_detectados.append(e.get('valor'))
        elif e.get('tipo') in ['MES_ANIO', 'MES_NUM_ANIO']:
            meses_detectados.append(e.get('mes'))
        elif e.get('tipo') == 'RANGO_MESES':
            meses_detectados.extend([e.get('mes_inicio'), e.get('mes_fin')])

    meses_detectados = [m for m in meses_detectados if m is not None]

    return len(set(meses_detectados)) > 1


df['tiene_entidad_temporal'] = df['entidades_temporales_reglas'].apply(tiene_entidad_temporal)
df['cantidad_entidades_temporales'] = df['entidades_temporales_reglas'].apply(len)
df['cantidad_meses_detectados'] = df['entidades_temporales_reglas'].apply(lambda x: contar_entidades_tipo(x, 'MES'))
df['registro_temporal_ambiguo'] = df['entidades_temporales_reglas'].apply(es_ambiguo_temporal)

df[
    [
        'OBSERVACION',
        'entidades_ner',
        'entidades_temporales_reglas',
        'tiene_entidad_temporal',
        'cantidad_entidades_temporales',
        'registro_temporal_ambiguo'
    ]
].head(20)

,OBSERVACION,entidades_ner,entidades_temporales_reglas,tiene_entidad_temporal,cantidad_entidades_temporales,registro_temporal_ambiguo
0,SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVI...,[],[],False,0,False
1,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[],[],False,0,False
2,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,[],[],False,0,False
3,SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE ...,[],[],False,0,False
4,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,[],[],False,0,False
5,SERVICIO DE MENSAJERIA PRESTADO DURANTE EL MES...,"[(microsoft, ORG)]",[],False,0,False
6,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[],[],False,0,False
7,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[],[],False,0,False
8,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[],[],False,0,False
9,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,[],[],False,0,False


## 4.7 Embeddings de observaciones

Los embeddings permiten convertir cada observación textual en una representación numérica. Esto facilita comparar observaciones entre sí, encontrar textos similares y agrupar registros con patrones comunes de facturación o desfase.

Para esta entrega se utiliza TF-IDF como representación vectorial base, por ser interpretable y suficiente para una primera aproximación funcional.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Vectorización TF-IDF como embeddings interpretables de texto
vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X_embeddings = vectorizer.fit_transform(df['observacion_normalizada'].fillna(''))

print('Dimensión de la matriz de embeddings:', X_embeddings.shape)

# Palabras o n-gramas más relevantes del vocabulario
vocabulario = vectorizer.get_feature_names_out()
vocabulario[:30]

Dimensión de la matriz de embeddings: (8011, 500)


array(['003', '003 17', '0111', '0111 corredor', '05', '05 meinca',
       '05 meinca2017', '072', '11', '113', '113 2020', '13', '14', '15',
       '15 de', '16', '17', '18', '18 0111', '20', '20 de', '2017',
       '2017 ent', '2020', '2020 contacto', '2023', '2024',
       '2024 contacto', '2024 contrato', '2024 de'], dtype=object)

## 4.8 Clustering con embeddings

Con los embeddings se pueden agrupar observaciones similares para identificar patrones recurrentes en la columna `OBSERVACION`.

In [15]:
from sklearn.cluster import KMeans

# Número inicial de grupos para exploración
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df['cluster_observacion'] = kmeans.fit_predict(X_embeddings)

# Resumen por cluster
resumen_clusters = df.groupby('cluster_observacion').agg(
    cantidad_registros=('OBSERVACION', 'count'),
    desfase_promedio=('desfase_meses', 'mean') if 'desfase_meses' in df.columns else ('OBSERVACION', 'count')
).reset_index()

resumen_clusters

,cluster_observacion,cantidad_registros,desfase_promedio
0,0,1364,1364
1,1,1561,1561
2,2,647,647
3,3,779,779
4,4,3660,3660


## 4.9 Ejemplos de observaciones por cluster

Se muestran ejemplos de observaciones agrupadas para interpretar los patrones textuales identificados.

In [16]:
for cluster in sorted(df['cluster_observacion'].unique()):
    print('\n============================')
    print('Cluster:', cluster)
    print('============================')
    ejemplos = df[df['cluster_observacion'] == cluster]['OBSERVACION'].dropna().head(5).tolist()
    for ej in ejemplos:
        print('-', ej)


Cluster: 0
- SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVIANA PRESTADOS EN EL MES DE DICIEMBRE SEGUN OC 4824117
- SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE DE CARGA LIVIANA  OC 9400069513
- SERVICIO TRANSPORTE TERRESTRE DE CARGA LIVIANA PRESTADOS EN DICIEMBRE 2023
- SERVICIO TRANSPORTE TERRESTRE DE CARGA LIVIANA PRESTADOS EN DICIEMBRE 2023
- SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVIANA PRESTADO EN EL MES DE DICIEMBRE 2023

Cluster: 1
- SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIANA PRESTADOS DURANTE EL MES DE DICIEMBRE SEGUN OC 4824136
- SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIANA PRESTADOS EN EL MES DICIEMBRE 2023 SEGÚN OC 4013
- SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIANA PRESTADOS EN EL MES DICIEMBRE 2023 SEGÚN OC 42953
- SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIANA PRESTADOS EN EL MES DICIEMBRE 2023 SEGÚN OC 99529
- SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIANA PRESTADOS EN EL MES DICIEMBRE 2023 SEGÚN OC 2817

Cluster: 2
- SERVICIO DE TRANSPORTE TERRES

## 5. Diccionario de meses

Se definen nombres completos y abreviaciones para identificar meses dentro del texto.

In [17]:
meses = {
    'enero': 1, 'ene': 1,
    'febrero': 2, 'feb': 2,
    'marzo': 3, 'mar': 3,
    'abril': 4, 'abr': 4,
    'mayo': 5, 'may': 5,
    'junio': 6, 'jun': 6,
    'julio': 7, 'jul': 7,
    'agosto': 8, 'ago': 8,
    'septiembre': 9, 'setiembre': 9, 'sep': 9, 'sept': 9,
    'octubre': 10, 'oct': 10,
    'noviembre': 11, 'nov': 11,
    'diciembre': 12, 'dic': 12
}

patron_meses = r'\b(' + '|'.join(meses.keys()) + r')\b'
patron_anio = r'\b(20\d{2})\b'

## 6. Extracción de mes y año

El algoritmo busca meses y años explícitos en la observación. Si el año no aparece, lo infiere usando la fecha de factura.

In [18]:
def extraer_mes(texto):
    coincidencias = re.findall(patron_meses, texto)
    if coincidencias:
        return meses.get(coincidencias[0])
    return np.nan

def extraer_anio(texto):
    coincidencias = re.findall(patron_anio, texto)
    if coincidencias:
        return int(coincidencias[0])
    return np.nan

df['mes_extraido'] = df['observacion_normalizada'].apply(extraer_mes)
df['anio_extraido'] = df['observacion_normalizada'].apply(extraer_anio)

df[['OBSERVACION', 'mes_extraido', 'anio_extraido']].head(20)

,OBSERVACION,mes_extraido,anio_extraido
0,SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVI...,12.0,NaN
1,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,NaN
2,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,12.0,2023.0
3,SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE ...,NaN,NaN
4,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,12.0,2023.0
5,SERVICIO DE MENSAJERIA PRESTADO DURANTE EL MES...,1.0,2024.0
6,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,2023.0
7,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,2023.0
8,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,2023.0
9,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,2023.0


## 7. Inferencia del año

Cuando el año no aparece en la observación, se usa la fecha de factura como referencia. Si el mes extraído es posterior al mes de factura, se asume que corresponde al año anterior.

In [19]:
df['FECHA'] = pd.to_datetime(df['FECHA'], errors='coerce')

def inferir_anio(row):
    if not pd.isna(row['anio_extraido']):
        return int(row['anio_extraido'])
    if pd.isna(row['FECHA']) or pd.isna(row['mes_extraido']):
        return np.nan
    anio_factura = row['FECHA'].year
    mes_factura = row['FECHA'].month
    mes_servicio = int(row['mes_extraido'])
    if mes_servicio > mes_factura:
        return anio_factura - 1
    return anio_factura

df['anio_servicio_estimado'] = df.apply(inferir_anio, axis=1)
df[['FECHA', 'OBSERVACION', 'mes_extraido', 'anio_extraido', 'anio_servicio_estimado']].head(20)

,FECHA,OBSERVACION,mes_extraido,anio_extraido,anio_servicio_estimado
0,2024-01-02,SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVI...,12.0,NaN,2023.0
1,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,NaN,2023.0
2,2024-01-02,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,12.0,2023.0,2023.0
3,2024-01-02,SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE ...,NaN,NaN,NaN
4,2024-01-02,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,12.0,2023.0,2023.0
5,2024-01-02,SERVICIO DE MENSAJERIA PRESTADO DURANTE EL MES...,1.0,2024.0,2024.0
6,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,2023.0,2023.0
7,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,2023.0,2023.0
8,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,2023.0,2023.0
9,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,12.0,2023.0,2023.0


## 8. Construcción de fecha de servicio y cálculo del desfase

In [20]:
def construir_fecha_servicio(row):
    if pd.isna(row['mes_extraido']) or pd.isna(row['anio_servicio_estimado']):
        return pd.NaT
    return pd.Timestamp(year=int(row['anio_servicio_estimado']), month=int(row['mes_extraido']), day=1)

df['fecha_servicio_extraida'] = df.apply(construir_fecha_servicio, axis=1)

df['desfase_dias'] = (df['FECHA'] - df['fecha_servicio_extraida']).dt.days
df['desfase_meses'] = ((df['FECHA'].dt.year - df['fecha_servicio_extraida'].dt.year) * 12 +
                       (df['FECHA'].dt.month - df['fecha_servicio_extraida'].dt.month))

df[['FECHA', 'OBSERVACION', 'fecha_servicio_extraida', 'desfase_dias', 'desfase_meses']].head(20)

,FECHA,OBSERVACION,fecha_servicio_extraida,desfase_dias,desfase_meses
0,2024-01-02,SERVICIO DE TRANSPORTE TERRESTRE DE CARGA LIVI...,2023-12-01,32.0,1.0
1,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2023-12-01,32.0,1.0
2,2024-01-02,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,2023-12-01,32.0,1.0
3,2024-01-02,SERVICIO DE MENSAJERIA Y TRANSPORTE TERRESTRE ...,NaT,NaN,NaN
4,2024-01-02,SERVICIO DE MENSAJERIA FIJA DEL MES DE DICIEMB...,2023-12-01,32.0,1.0
5,2024-01-02,SERVICIO DE MENSAJERIA PRESTADO DURANTE EL MES...,2024-01-01,1.0,0.0
6,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2023-12-01,32.0,1.0
7,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2023-12-01,32.0,1.0
8,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2023-12-01,32.0,1.0
9,2024-01-02,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2023-12-01,32.0,1.0


## 9. Resumen de extracción

In [21]:
total_registros = len(df)
registros_con_fecha = df['fecha_servicio_extraida'].notna().sum()
registros_sin_fecha = df['fecha_servicio_extraida'].isna().sum()
porcentaje_exito = registros_con_fecha / total_registros * 100

resumen = pd.DataFrame({
    'Métrica': ['Total registros', 'Registros con fecha extraída', 'Registros sin fecha extraída', '% extracción exitosa'],
    'Valor': [total_registros, registros_con_fecha, registros_sin_fecha, round(porcentaje_exito, 2)]
})

resumen

,Métrica,Valor
0,Total registros,8011.00
1,Registros con fecha extraída,7294.00
2,Registros sin fecha extraída,717.00
3,% extracción exitosa,91.05


## 10. Análisis de patrones de desfase

In [22]:
df_validos = df[df['fecha_servicio_extraida'].notna()].copy()

print('Desfase promedio en meses:', round(df_validos['desfase_meses'].mean(), 2))
print('Mediana del desfase en meses:', round(df_validos['desfase_meses'].median(), 2))
print('Registros con desfase mayor a 1 mes:', (df_validos['desfase_meses'] > 1).sum())

df_validos['desfase_meses'].value_counts().sort_index().head(20)

Desfase promedio en meses: 0.91
Mediana del desfase en meses: 1.0
Registros con desfase mayor a 1 mes: 951


,count
desfase_meses,
-587.0,1
-11.0,36
-10.0,16
-1.0,2
0.0,1064
1.0,5224
2.0,749
3.0,113
4.0,42


## 11. Guardar resultados

Se generan dos archivos: resultados completos y resumen de métricas.

In [23]:
archivo_resultados = 'resultados_extraccion_pln.xlsx'
archivo_resumen = 'resumen_metricas_pln.xlsx'

df.to_excel(archivo_resultados, index=False)
resumen.to_excel(archivo_resumen, index=False)

print('Archivos generados:')
print(archivo_resultados)
print(archivo_resumen)

Archivos generados:
resultados_extraccion_pln.xlsx
resumen_metricas_pln.xlsx


## 13. Evaluación correcta del modelo PLN

Las métricas **Precision**, **Recall** y **F1-Score** requieren una muestra validada manualmente. No es correcto calcularlas comparando el resultado del algoritmo contra una variable creada por el mismo algoritmo, porque eso genera métricas artificialmente perfectas.

Por esta razón, el notebook separa dos tipos de resultados:

1. **Cobertura automática de extracción:** porcentaje de registros donde el algoritmo logró extraer una fecha.
2. **Métricas supervisadas reales:** Precision, Recall, F1-Score y MAE, calculadas únicamente si existe validación manual.

In [24]:
# =============================================================
# 13.1 MÉTRICAS DESCRIPTIVAS DE COBERTURA
# =============================================================
# Estas métricas NO reemplazan Precision, Recall ni F1.
# Solo indican qué tanto logró procesar el algoritmo.

total_registros = len(df)
registros_con_fecha_extraida = df['fecha_servicio_extraida'].notna().sum()
registros_sin_fecha_extraida = df['fecha_servicio_extraida'].isna().sum()
cobertura_extraccion = registros_con_fecha_extraida / total_registros

tabla_cobertura = pd.DataFrame({
    'Métrica': [
        'Total de registros',
        'Registros con fecha extraída',
        'Registros sin fecha extraída',
        'Cobertura de extracción'
    ],
    'Valor': [
        total_registros,
        registros_con_fecha_extraida,
        registros_sin_fecha_extraida,
        f'{cobertura_extraccion:.2%}'
    ]
})

tabla_cobertura

,Métrica,Valor
0,Total de registros,8011
1,Registros con fecha extraída,7294
2,Registros sin fecha extraída,717
3,Cobertura de extracción,91.05%


## 13.2 Crear muestra para validación manual

Para calcular métricas reales, se debe validar manualmente una muestra de observaciones. La siguiente celda genera un archivo Excel con una muestra aleatoria y columnas para diligenciar manualmente.

Columnas a completar:

- `contiene_fecha_real`: 1 si la observación realmente contiene una fecha de servicio; 0 si no la contiene.
- `fecha_servicio_real`: fecha real identificada manualmente, en formato AAAA-MM-DD.
- `desfase_real_meses`: desfase real validado manualmente.

In [28]:
# =============================================================
# GENERAR MUESTRA PARA VALIDACIÓN MANUAL BINARIA
# =============================================================
# SOLO se validará:
#
# contiene_fecha_real
#
# 1 = la observación contiene una fecha temporal
# 0 = la observación NO contiene una fecha temporal
# =============================================================

# Tamaño de muestra sugerido
n_muestra = min(200, len(df))

# Selección aleatoria reproducible
muestra_validacion = (

    df
    .sample(
        n=n_muestra,
        random_state=42
    )
    .copy()

)

# =============================================================
# COLUMNA A DILIGENCIAR MANUALMENTE
# =============================================================

muestra_validacion['contiene_fecha_real'] = ''

# =============================================================
# COLUMNAS A EXPORTAR
# =============================================================

columnas_validacion = [

    'OBSERVACION',
    'FECHA',
    'observacion_normalizada',
    'tokens_observacion',
    'mes_extraido',
    'anio_servicio_estimado',
    'fecha_servicio_extraida',
    'cantidad_entidades_temporales',
    'registro_temporal_ambiguo',
    'contiene_fecha_real'

]

# Mantener únicamente columnas existentes
muestra_validacion = (

    muestra_validacion[
        [
            c
            for c in columnas_validacion
            if c in muestra_validacion.columns
        ]
    ]

)

# =============================================================
# EXPORTAR ARCHIVO
# =============================================================

archivo_muestra = (
    'muestra_validacion_manual_pln.xlsx'
)

muestra_validacion.to_excel(
    archivo_muestra,
    index=False
)

# =============================================================
# RESUMEN
# =============================================================

print(
    'Archivo generado para validación manual:',
    archivo_muestra
)

print(
    'Cantidad de registros en muestra:',
    len(muestra_validacion)
)

print(
    'Columnas exportadas:',
    list(muestra_validacion.columns)
)

muestra_validacion.head(10)

Archivo generado para validación manual: muestra_validacion_manual_pln.xlsx
Cantidad de registros en muestra: 200
Columnas exportadas: ['OBSERVACION', 'FECHA', 'observacion_normalizada', 'tokens_observacion', 'mes_extraido', 'anio_servicio_estimado', 'fecha_servicio_extraida', 'cantidad_entidades_temporales', 'registro_temporal_ambiguo', 'contiene_fecha_real']


,OBSERVACION,FECHA,observacion_normalizada,tokens_observacion,mes_extraido,anio_servicio_estimado,fecha_servicio_extraida,cantidad_entidades_temporales,registro_temporal_ambiguo,contiene_fecha_real
554,SERVICIO DE MENSAJERIA NACIONAL CARGA LIVIANA ...,2024-03-12,servicio de mensajeria nacional carga liviana ...,"[servicio, de, mensajeria, nacional, carga, li...",2.0,2024.0,2024-02-01,0,False,
3872,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2025-04-16,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",3.0,2025.0,2025-03-01,0,False,
3103,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2025-01-22,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",12.0,2024.0,2024-12-01,0,False,
7049,SERVICIO TERRESTRE DE CARGA LIVIANA NIVEL NACI...,2025-06-25,servicio terrestre de carga liviana nivel naci...,"[servicio, terrestre, de, carga, liviana, nive...",5.0,2025.0,2025-05-01,0,False,
7861,SERVICIO TERRESTRE CARGA LIVIANA NIVEL NACIONA...,2025-09-18,servicio terrestre carga liviana nivel naciona...,"[servicio, terrestre, carga, liviana, nivel, n...",8.0,2025.0,2025-08-01,0,False,
1385,TRANSPORTE DE MENSAJERIA NACIONAL MES MAYO 49...,2024-06-26,transporte de mensajeria nacional mes mayo 490...,"[transporte, de, mensajeria, nacional, mes, ma...",5.0,2024.0,2024-05-01,0,False,
3362,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2025-02-19,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",1.0,2025.0,2025-01-01,0,False,
2264,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2024-10-10,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",10.0,2024.0,2024-10-01,0,False,
5633,SERVICIO MENSAJERIA TERRESTRE CARGA LIVIANA (N...,2025-02-01,servicio mensajeria terrestre carga liviana ne...,"[servicio, mensajeria, terrestre, carga, livia...",NaN,NaN,NaT,0,False,
5831,SERVICIO CORRESPONDENCIA DIGITALIZACION FACTUR...,2025-02-26,servicio correspondencia digitalizacion factur...,"[servicio, correspondencia, digitalizacion, fa...",2.0,2025.0,2025-02-01,0,False,


## 13.3 Cálculo real de Precision, Recall, F1-Score y MAE

Después de diligenciar manualmente el archivo `muestra_validacion_manual_pln.xlsx`, se debe volver a subir a Colab y ejecutar la siguiente celda.

Esta celda sí calcula las métricas reales.

In [30]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# =============================================================
# VALIDACIÓN MANUAL BINARIA
# =============================================================
# SOLO se valida:
#
# contiene_fecha_real
#
# 1 = la observación sí contiene fecha
# 0 = la observación NO contiene fecha
#
# El algoritmo predice:
#
# 1 = logró extraer fecha
# 0 = no logró extraer fecha
# =============================================================

usar_validacion_manual = True

if usar_validacion_manual:

    from google.colab import files

    uploaded_validacion = files.upload()

    archivo_validado = list(
        uploaded_validacion.keys()
    )[0]

    df_validado = pd.read_excel(
        archivo_validado
    )

    # =========================================================
    # PREDICCIÓN DEL ALGORITMO
    # =========================================================

    df_validado['pred_fecha_detectada'] = (

        df_validado['fecha_servicio_extraida']
        .notna()
        .astype(int)

    )

    # =========================================================
    # VERDAD REAL MANUAL
    # =========================================================

    df_validado['contiene_fecha_real'] = (

        pd.to_numeric(
            df_validado['contiene_fecha_real'],
            errors='coerce'
        )

    )

    # =========================================================
    # ELIMINAR REGISTROS VACÍOS
    # =========================================================

    eval_clasificacion = (

        df_validado
        .dropna(
            subset=['contiene_fecha_real']
        )
        .copy()

    )

    # =========================================================
    # VARIABLES OBJETIVO
    # =========================================================

    y_true = (

        eval_clasificacion['contiene_fecha_real']
        .astype(int)

    )

    y_pred = (

        eval_clasificacion['pred_fecha_detectada']
        .astype(int)

    )

    # =========================================================
    # MÉTRICAS
    # =========================================================

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    # =========================================================
    # TABLA FINAL
    # =========================================================

    tabla_metricas_reales = pd.DataFrame({

        'Métrica': [

            'Accuracy',
            'Precision',
            'Recall / Exhaustividad',
            'F1-Score'

        ],

        'Descripción': [

            'Porcentaje total de predicciones correctas',
            'Proporción de detecciones correctas sobre las detectadas',
            'Proporción de fechas reales detectadas',
            'Balance entre Precision y Recall'

        ],

        'Valor': [

            round(accuracy, 4),
            round(precision, 4),
            round(recall, 4),
            round(f1, 4)

        ]

    })

else:

    tabla_metricas_reales = pd.DataFrame({

        'Métrica': [

            'Accuracy',
            'Precision',
            'Recall / Exhaustividad',
            'F1-Score'

        ],

        'Valor': [

            'Pendiente validación manual',
            'Pendiente validación manual',
            'Pendiente validación manual',
            'Pendiente validación manual'

        ]

    })

tabla_metricas_reales

Saving muestra_validacion_manual_pln (1) (2).xlsx to muestra_validacion_manual_pln (1) (2).xlsx


,Métrica,Descripción,Valor
0,Accuracy,Porcentaje total de predicciones correctas,0.8650
1,Precision,Proporción de detecciones correctas sobre las ...,0.8492
2,Recall / Exhaustividad,Proporción de fechas reales detectadas,1.0000
3,F1-Score,Balance entre Precision y Recall,0.9184


## 13.4 Interpretación correcta

Si todavía no existe validación manual, el resultado que se debe reportar es la **cobertura de extracción**, no Precision, Recall ni F1-Score.

Las métricas supervisadas deben quedar como pendientes hasta que se complete la muestra validada manualmente.

## 13. Evaluación correcta del pipeline PLN

En esta sección se separan las métricas que **sí se pueden calcular automáticamente** de las métricas que requieren **validación manual**.

No se deben calcular `Precision`, `Recall` ni `F1-Score` usando como verdad real una variable creada por el mismo algoritmo, porque eso genera resultados artificialmente perfectos como 1.0.

Por eso, primero se reportan métricas descriptivas de cobertura y luego se deja el bloque para calcular las métricas supervisadas reales cuando exista una muestra validada manualmente.

In [31]:
# =============================================================
# MÉTRICAS DESCRIPTIVAS DEL PIPELINE
# Estas métricas NO requieren validación manual
# =============================================================

import time
from sklearn.metrics import silhouette_score

total_registros = len(df)
registros_con_fecha_extraida = df['fecha_servicio_extraida'].notna().sum()
registros_sin_fecha_extraida = df['fecha_servicio_extraida'].isna().sum()
cobertura_extraccion = registros_con_fecha_extraida / total_registros

# Registros ambiguos: observaciones donde aparecen dos o más meses diferentes
def detectar_ambiguedad(texto):
    coincidencias = re.findall(patron_meses, str(texto))
    return len(set(coincidencias)) > 1

df['registro_ambiguo'] = df['observacion_normalizada'].apply(detectar_ambiguedad)
registros_ambiguos = df['registro_ambiguo'].sum()
porcentaje_ambiguos = registros_ambiguos / total_registros

# Tiempo de procesamiento de la tokenización
inicio = time.time()
_ = df['observacion_normalizada'].apply(tokenizar_texto)
fin = time.time()
tiempo_total = fin - inicio
registros_por_segundo = total_registros / tiempo_total if tiempo_total > 0 else 0

# Diversidad léxica
todos_tokens = [token for lista in df['tokens_observacion'] for token in lista]
tokens_unicos = len(set(todos_tokens))
total_tokens = len(todos_tokens)
diversidad_lexica = tokens_unicos / total_tokens if total_tokens > 0 else 0
promedio_tokens = df['cantidad_tokens'].mean()

# Silhouette Score si existe clustering
try:
    sil_score = silhouette_score(X_embeddings, df['cluster_observacion'])
except Exception:
    sil_score = None

tabla_metricas_descriptivas = pd.DataFrame({
    'Métrica': [
        'Total registros',
        'Registros con fecha extraída',
        'Registros sin fecha extraída',
        'Cobertura de extracción',
        'Registros ambiguos',
        '% registros ambiguos',
        'Tiempo procesamiento tokenización (seg)',
        'Registros por segundo',
        'Promedio tokens por observación',
        'Diversidad léxica',
        'Silhouette Score clustering'
    ],
    'Valor': [
        total_registros,
        registros_con_fecha_extraida,
        registros_sin_fecha_extraida,
        f'{cobertura_extraccion:.2%}',
        registros_ambiguos,
        f'{porcentaje_ambiguos:.2%}',
        round(tiempo_total, 4),
        round(registros_por_segundo, 2),
        round(promedio_tokens, 2),
        round(diversidad_lexica, 4),
        round(sil_score, 4) if sil_score is not None else 'No calculado'
    ]
})

tabla_metricas_descriptivas


,Métrica,Valor
0,Total registros,8011
1,Registros con fecha extraída,7294
2,Registros sin fecha extraída,717
3,Cobertura de extracción,91.05%
4,Registros ambiguos,732
5,% registros ambiguos,9.14%
6,Tiempo procesamiento tokenización (seg),0.0439
7,Registros por segundo,182416.49
8,Promedio tokens por observación,12.85
9,Diversidad léxica,0.0442


## 14. Generar muestra para validación manual

Para calcular `Precision`, `Recall`, `F1-Score` y `MAE` de forma correcta, se debe validar manualmente una muestra.

La siguiente celda crea un archivo Excel con 200 registros para que una persona complete:

- `contiene_fecha_real`: 1 si la observación realmente contiene fecha de servicio, 0 si no.
- `fecha_servicio_real`: fecha real identificada manualmente.
- `desfase_real_meses`: desfase real validado manualmente.

In [32]:
n_muestra = min(200, len(df))
muestra_validacion = df.sample(n=n_muestra, random_state=42).copy()

muestra_validacion['contiene_fecha_real'] = ''
muestra_validacion['fecha_servicio_real'] = ''
muestra_validacion['desfase_real_meses'] = ''

columnas_validacion = [
    'OBSERVACION',
    'FECHA',
    'observacion_normalizada',
    'tokens_observacion',
    'mes_extraido',
    'anio_servicio_estimado',
    'fecha_servicio_extraida',
    'desfase_meses',
    'contiene_fecha_real',
    'fecha_servicio_real',
    'desfase_real_meses'
]

muestra_validacion = muestra_validacion[[c for c in columnas_validacion if c in muestra_validacion.columns]]

archivo_muestra = 'muestra_validacion_manual_pln.xlsx'
muestra_validacion.to_excel(archivo_muestra, index=False)

print('Archivo generado:', archivo_muestra)
muestra_validacion.head(10)


Archivo generado: muestra_validacion_manual_pln.xlsx


,OBSERVACION,FECHA,observacion_normalizada,tokens_observacion,mes_extraido,anio_servicio_estimado,fecha_servicio_extraida,desfase_meses,contiene_fecha_real,fecha_servicio_real,desfase_real_meses
554,SERVICIO DE MENSAJERIA NACIONAL CARGA LIVIANA ...,2024-03-12,servicio de mensajeria nacional carga liviana ...,"[servicio, de, mensajeria, nacional, carga, li...",2.0,2024.0,2024-02-01,1.0,,,
3872,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2025-04-16,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",3.0,2025.0,2025-03-01,1.0,,,
3103,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2025-01-22,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",12.0,2024.0,2024-12-01,1.0,,,
7049,SERVICIO TERRESTRE DE CARGA LIVIANA NIVEL NACI...,2025-06-25,servicio terrestre de carga liviana nivel naci...,"[servicio, terrestre, de, carga, liviana, nive...",5.0,2025.0,2025-05-01,1.0,,,
7861,SERVICIO TERRESTRE CARGA LIVIANA NIVEL NACIONA...,2025-09-18,servicio terrestre carga liviana nivel naciona...,"[servicio, terrestre, carga, liviana, nivel, n...",8.0,2025.0,2025-08-01,1.0,,,
1385,TRANSPORTE DE MENSAJERIA NACIONAL MES MAYO 49...,2024-06-26,transporte de mensajeria nacional mes mayo 490...,"[transporte, de, mensajeria, nacional, mes, ma...",5.0,2024.0,2024-05-01,1.0,,,
3362,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2025-02-19,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",1.0,2025.0,2025-01-01,1.0,,,
2264,SERVICIO DE MENSAJERIA NACIONAL DE CARGA LIVIA...,2024-10-10,servicio de mensajeria nacional de carga livia...,"[servicio, de, mensajeria, nacional, de, carga...",10.0,2024.0,2024-10-01,0.0,,,
5633,SERVICIO MENSAJERIA TERRESTRE CARGA LIVIANA (N...,2025-02-01,servicio mensajeria terrestre carga liviana ne...,"[servicio, mensajeria, terrestre, carga, livia...",NaN,NaN,NaT,NaN,,,
5831,SERVICIO CORRESPONDENCIA DIGITALIZACION FACTUR...,2025-02-26,servicio correspondencia digitalizacion factur...,"[servicio, correspondencia, digitalizacion, fa...",2.0,2025.0,2025-02-01,0.0,,,


## 15. Métricas supervisadas reales

Después de diligenciar la muestra manual, cambia `usar_validacion_manual = True`, sube el archivo validado y ejecuta la celda.

Solo en ese momento se calculan correctamente:

- Precision
- Recall / Exhaustividad
- F1-Score
- Exact Match Accuracy
- MAE

In [33]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, mean_absolute_error

usar_validacion_manual = False  # Cambiar a True cuando ya tengas el Excel validado manualmente

if usar_validacion_manual:
    from google.colab import files
    uploaded_validacion = files.upload()
    archivo_validado = list(uploaded_validacion.keys())[0]
    df_validado = pd.read_excel(archivo_validado)

    df_validado['fecha_servicio_extraida'] = pd.to_datetime(df_validado['fecha_servicio_extraida'], errors='coerce')
    df_validado['fecha_servicio_real'] = pd.to_datetime(df_validado['fecha_servicio_real'], errors='coerce')
    df_validado['contiene_fecha_real'] = pd.to_numeric(df_validado['contiene_fecha_real'], errors='coerce')
    df_validado['desfase_real_meses'] = pd.to_numeric(df_validado['desfase_real_meses'], errors='coerce')
    df_validado['desfase_meses'] = pd.to_numeric(df_validado['desfase_meses'], errors='coerce')

    df_validado['pred_fecha_detectada'] = df_validado['fecha_servicio_extraida'].notna().astype(int)

    eval_clf = df_validado.dropna(subset=['contiene_fecha_real']).copy()
    y_true = eval_clf['contiene_fecha_real'].astype(int)
    y_pred = eval_clf['pred_fecha_detectada'].astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    eval_exact = df_validado.dropna(subset=['fecha_servicio_real', 'fecha_servicio_extraida']).copy()
    exact_match = (eval_exact['fecha_servicio_real'] == eval_exact['fecha_servicio_extraida']).mean() if len(eval_exact) > 0 else None

    eval_mae = df_validado.dropna(subset=['desfase_real_meses', 'desfase_meses']).copy()
    mae = mean_absolute_error(eval_mae['desfase_real_meses'], eval_mae['desfase_meses']) if len(eval_mae) > 0 else None

    tabla_metricas_supervisadas = pd.DataFrame({
        'Métrica': [
            'Precision',
            'Recall / Exhaustividad',
            'F1-Score',
            'Exact Match Accuracy',
            'Error Absoluto Medio (MAE)'
        ],
        'Valor': [
            round(precision, 4),
            round(recall, 4),
            round(f1, 4),
            round(exact_match, 4) if exact_match is not None else 'No calculado',
            round(mae, 4) if mae is not None else 'No calculado'
        ]
    })
else:
    tabla_metricas_supervisadas = pd.DataFrame({
        'Métrica': [
            'Precision',
            'Recall / Exhaustividad',
            'F1-Score',
            'Exact Match Accuracy',
            'Error Absoluto Medio (MAE)'
        ],
        'Valor': [
            'Pendiente validación manual',
            'Pendiente validación manual',
            'Pendiente validación manual',
            'Pendiente validación manual',
            'Pendiente validación manual'
        ]
    })

tabla_metricas_supervisadas


,Métrica,Valor
0,Precision,Pendiente validación manual
1,Recall / Exhaustividad,Pendiente validación manual
2,F1-Score,Pendiente validación manual
3,Exact Match Accuracy,Pendiente validación manual
4,Error Absoluto Medio (MAE),Pendiente validación manual


## 12. Descargar archivos generados

In [34]:
files.download(archivo_resultados)
files.download(archivo_resumen)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>